# Домашнее задание № 2. Мешок слов

## Задание 1 (3 балла)

У векторайзеров в sklearn есть встроенная токенизация на регулярных выражениях. Найдите способо заменить её на кастомную токенизацию

Обучите векторайзер с дефолтной токенизацией и с токенизацией razdel.tokenize. Обучите классификатор (любой) с каждым из векторизаторов. Сравните метрики и выберете победителя. 

(в вашей тетрадке должен быть код обучения и все метрики; если вы сдаете в .py файлах то сохраните полученные метрики в отдельном файле или в комментариях)

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from IPython.display import Image
from IPython.core.display import HTML 

c:\Users\schig\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.4)
  from scipy.sparse import csr_matrix, issparse


In [2]:
data = pd.read_csv("C:/Users/schig/Downloads/labeled.csv")
data.head()

,comment,toxic
0,"Верблюдов-то за что? Дебилы, бл...\n",1.0
1,"Хохлы, это отдушина затюканого россиянина, мол...",1.0
2,Собаке - собачья смерть\n,1.0
3,"Страницу обнови, дебил. Это тоже не оскорблени...",1.0
4,"тебя не убедил 6-страничный пдф в том, что Скр...",1.0


In [3]:
X = data['comment']
y = data['toxic']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def razdel_tokenizer(text):
    from razdel import tokenize 
    return [t.text for t in tokenize(text)]

In [4]:
vectorizer_default = CountVectorizer()
X_train_default = vectorizer_default.fit_transform(X_train)
X_test_default = vectorizer_default.transform(X_test)

vectorizer_razdel = CountVectorizer(tokenizer=razdel_tokenizer, preprocessor=None)
X_train_razdel = vectorizer_razdel.fit_transform(X_train)
X_test_razdel = vectorizer_razdel.transform(X_test)

c:\Users\schig\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [10]:
clf_default = LogisticRegression(max_iter=1000)
clf_default.fit(X_train_default, y_train)

clf_razdel = LogisticRegression(max_iter=1000)
clf_razdel.fit(X_train_razdel, y_train)

y_pred_default = clf_default.predict(X_test_default)
y_pred_razdel = clf_razdel.predict(X_test_razdel)

def get_metrics(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"\n{name}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    return {"model": name, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

metrics_default = get_metrics(y_test, y_pred_default, "Default tokenizer")
metrics_razdel = get_metrics(y_test, y_pred_razdel, "Razdel tokenizer")

results = pd.DataFrame([metrics_default, metrics_razdel])
print("\nСравнение метрик:")
print(results)

best_model = results.loc[results['f1'].idxmax()]
print(f"\nПобедитель: {best_model['model']} (F1 = {best_model['f1']:.4f})")

results.to_csv("metrics_comparison.csv", index=False)


Default tokenizer
Accuracy:  0.8515
Precision: 0.8575
Recall:    0.6674
F1-score:  0.7506

Razdel tokenizer
Accuracy:  0.8592
Precision: 0.8481
Recall:    0.7057
F1-score:  0.7704

Сравнение метрик:
               model  accuracy  precision    recall        f1
0  Default tokenizer  0.851544   0.857523  0.667358  0.750583
1   Razdel tokenizer  0.859174   0.848070  0.705699  0.770362

Победитель: Razdel tokenizer (F1 = 0.7704)


## Задание 2 (3 балла)

Обучите 2 любых разных классификатора из семинара. Предскажите токсичность для текстов из тестовой выборки (используйте одну и ту же выборку для обоих классификаторов) и найдите 10 самых токсичных для каждого из классификаторов. Сравните получаемые тексты - какие тексты совпадают, какие отличаются, правда ли тексты токсичные?

Требования к моделям:   
а) один классификатор должен использовать CountVectorizer, другой TfidfVectorizer  
б) у векторазера должны быть вручную заданы как минимум 5 параметров (можно ставить разные параметры tfidfvectorizer и countvectorizer)  
в) у классификатора должно быть задано вручную как минимум 2 параметра (по возможности)  
г)  f1 мера каждого из классификаторов должна быть минимум 0.75  

*random_seed не считается за параметр

In [6]:
data = pd.read_csv("C:/Users/schig/Downloads/labeled.csv")
X = data['comment']
y = data['toxic']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def razdel_tokenizer(text):
    from razdel import tokenize 
    return [t.text for t in tokenize(text)]

In [7]:
count_vect = CountVectorizer(
    lowercase=True,
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1,2),
    stop_words='english' 
)

X_train_count = count_vect.fit_transform(X_train)
X_test_count = count_vect.transform(X_test)

clf_logreg = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='liblinear',   
    random_state=42
)
clf_logreg.fit(X_train_count, y_train)
y_pred_count = clf_logreg.predict(X_test_count)

tfidf_vect = TfidfVectorizer(
    lowercase=True,
    max_features=6000,
    min_df=3,
    max_df=0.9,
    ngram_range=(1,3),
    norm='l2',
    sublinear_tf=True
)

X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)

clf_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
clf_rf.fit(X_train_tfidf, y_train)
y_pred_tfidf = clf_rf.predict(X_test_tfidf)

def print_metrics(y_true, y_pred, name):
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    print(name)    
    print(f"F1-score: {f1:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    return f1

f1_logreg = print_metrics(y_test, y_pred_count, "LogisticRegression + CountVectorizer")
f1_rf = print_metrics(y_test, y_pred_tfidf, "RandomForest + TfidfVectorizer")

probs_logreg = clf_logreg.predict_proba(X_test_count)[:,1]
top10_logreg_idx = probs_logreg.argsort()[-10:][::-1]
top10_logreg = X_test.iloc[top10_logreg_idx]

probs_rf = clf_rf.predict_proba(X_test_tfidf)[:,1]
top10_rf_idx = probs_rf.argsort()[-10:][::-1]
top10_rf = X_test.iloc[top10_rf_idx]

print("\nTop 10 токсичных комментариев (LogisticRegression):")
for i, txt in enumerate(top10_logreg, 1):
    print(f"{i}. {txt[:150]}")  

print("\nTop 10 токсичных комментариев (RandomForest):")
for i, txt in enumerate(top10_rf, 1):
    print(f"{i}. {txt[:150]}")

set_logreg = set(top10_logreg)
set_rf = set(top10_rf)

intersection = set_logreg.intersection(set_rf)
diff_logreg = set_logreg - set_rf
diff_rf = set_rf - set_logreg

print(f"\nСовпадают: {len(intersection)} комментариев")
print(f"Только в LogReg: {len(diff_logreg)}")
print(f"Только в RF: {len(diff_rf)}")

LogisticRegression + CountVectorizer
F1-score: 0.7194
Accuracy: 0.8266
Precision: 0.7846
Recall: 0.6642
RandomForest + TfidfVectorizer
F1-score: 0.6742
Accuracy: 0.7402
Precision: 0.5810
Recall: 0.8031

Top 10 токсичных комментариев (LogisticRegression):
1. Блеаадь, как же обидно когда создаешь тред, пародируешь речь ватников, случайно употребив слово из скрипта, и он скрывается у всех. Пересоздаю. Я мног
2. По мексикански Флаг: Ублюдок, мать твою, а ну иди сюда говно собачье, решил меня поднять? Ты, засранец вонючий, мать твою, а? Ну иди сюда, попробуй ме
3. Какие блять передергивания? Ты дебил блять зашел на шок-доску и удивляешься что над тобой издеваются. Тут нет твоих друзей, рачье тупорылое, тут тебя 
4. та ну, хуйня это все про джентельменство . меня в свое время эти брачные игрища так сильно заебали, что я вообще перестал париться и начал себя вести 
5. В role reversal треде fet поселился шизик, портящий всем живущим там картину. Последние несколько тредов он занимается такими 

## Задание 3 (4 балла - 1 балл за каждый классификатор)

Для классификаторов Logistic Regression, Decision Trees, Naive Bayes, RandomForest найдите способ извлечь важность признаков для предсказания токсичного класса. Сопоставьте полученные числа со словами (или нграммами) в словаре и найдите топ - 5 "токсичных" слов для каждого из классификаторов. 

Важное требование: в топе не должно быть стоп-слов. Для этого вам нужно будет правильным образом настроить векторизацию. 
Также как и в предыдущем задании у классификаторов должно быть задано вручную как минимум 2 параметра (по возможности, f1 мера каждого из классификаторов должна быть минимум 0.75

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.feature_selection import mutual_info_classif

data = pd.read_csv("C:/Users/schig/Downloads/labeled.csv")
X = data['comment']
y = data['toxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=10000,
    min_df=3,
    max_df=0.9,
    ngram_range=(1, 2),
    sublinear_tf=True,
    norm="l2"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
feature_names = np.array(tfidf.get_feature_names_out())

classifiers = {
    "LogisticRegression": LogisticRegression(C=1.0, solver="liblinear", max_iter=1000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=20, min_samples_split=5, random_state=42),
    "NaiveBayes": MultinomialNB(alpha=0.5, fit_prior=True),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, class_weight="balanced")
}

def evaluate_and_get_top_words(model, X_train, X_test, y_train, y_test, feature_names):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    if hasattr(model, "coef_"):
        importances = model.coef_[0]
    elif hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    elif isinstance(model, MultinomialNB):
        importances = model.feature_log_prob_[1, :] - model.feature_log_prob_[0, :]
    else:
        return f1, []

    if hasattr(model, "coef_") or isinstance(model, MultinomialNB):
        mask = importances > 0
        importances = importances[mask]
        words_filtered = feature_names[mask]
    else:
        words_filtered = feature_names

    top_indices = np.argsort(importances)[::-1]
    neutral_like = {"ты", "тебя", "тебе", "он", "она", "но", "не", "очень", "если", "как", "что", "это", "просто", "его", "так", "есть", "было", "можно"}
    top_words = []

    for idx in top_indices:
        word = words_filtered[idx]
        if (word.lower() not in ENGLISH_STOP_WORDS and word.lower() not in neutral_like and word.isalpha() and len(word) > 2):
            top_words.append((word, importances[idx]))
        if len(top_words) >= 5:
            break

    return f1, top_words

results = {}
for name, model in classifiers.items():
    f1, top_words = evaluate_and_get_top_words(model, X_train_tfidf, X_test_tfidf, y_train, y_test, feature_names)
    results[name] = {"f1": f1, "top_words": top_words}

for name in ["LogisticRegression", "DecisionTree", "NaiveBayes", "RandomForest"]:
    info = results[name]
    print(f"\n{name}: F1 = {info['f1']:.4f}")
    print("Top-5 токсичных слов/фраз:")
    for w, v in info['top_words']:
        print(f"  {w:20s} ({v:.4f})")




LogisticRegression: F1 = 0.6872
Top-5 токсичных слов/фраз:
  хохлов               (4.5973)
  хохлы                (4.4855)
  блядь                (3.0818)
  быдло                (2.9868)
  нахуй                (2.9283)

DecisionTree: F1 = 0.4293
Top-5 токсичных слов/фраз:
  хохлы                (0.0433)
  хохлов               (0.0335)
  блядь                (0.0207)
  нахуй                (0.0161)
  быдло                (0.0117)

NaiveBayes: F1 = 0.7119
Top-5 токсичных слов/фраз:
  хохлов               (4.7329)
  хохлы                (4.2336)
  сука                 (4.1639)
  дебил                (3.9276)
  хохол                (3.8044)

RandomForest: F1 = 0.6646
Top-5 токсичных слов/фраз:
  хохлы                (0.0163)
  хохлов               (0.0098)
  пиздец               (0.0092)
  блядь                (0.0086)
  нахуй                (0.0083)


## Задание 4* (2 балла)

Составьте ансамблевый классификатор вручную. Используя один из парных датасетов в glue (например, mnli) обучите как минимум 10 разных классификаторов (так как количество алгоритмов меньше 10, используйте разные комбинации параметров в векторайзерах и в классификаторах, например, tfidf + logreg и countvectorizer+logreg). Чем сильнее каждый классификатор отличается от другого, тем лучше.

Вместо стандартного разбиения на train и test, разбейте выборку на 3 части: train, dev и test. Используйте train для обучения 10 классификаторов. Сделайте предсказания всеми классификаторами для dev и test датасетов. Используя объединенные предсказания всех классификаторов на dev части как признаки, обучите еще один общий классификатор. Сделайте предсказания общим классификатором на test части (опять же используйте предсказания 10 классификаторов как признаки) и рассчитайте качество итоговой классификации.

Также отдельно оцените качество на test части для каждого из 10 классификаторов. Сравните с общей оценкой. Превосходит ли общий классификатор в качестве самый лучший из отдельных классификаторов?